In [ ]:
# !pip install torch_geometric

In [ ]:
# !pip install torch-scatter -f https://data.pyg.org/whl/torch-2.5.1+cu121.html

In [ ]:
# import all libraries needed downstream
import os
import gc
import wandb
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
import scipy
from sklearn.metrics import mean_squared_error
import math
import networkx as nx
import seaborn as sns
import time
from torch.nn import Linear
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import degree
from torch_geometric.nn import ChebConv, GraphConv, GCNConv, TAGConv, GATConv
from torch_geometric.data import Data
from torch.utils.data import TensorDataset
from torch_geometric.loader import DataLoader
import torch_scatter
from typing import Dict, Tuple, List, Optional
import matplotlib.pyplot as plt
from torch_scatter import scatter_softmax, scatter_sum
import scipy.sparse as sp
import scipy.sparse.linalg as spla
from torch.amp import autocast, GradScaler

In [ ]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
!nvidia-smi

In [ ]:
system_size = 118

In [ ]:
pgl_train_data = np.load(f'/home/oarowolo/workfile/OPFData/data/PGLearn/nminusone_topology/{system_size}_ieee-nminus1_train.npz')

# Get all keys
print("Available keys in the dataset:")
for key in pgl_train_data.files:
    # Print the key and its array shape
    print(f"{key}: shape {pgl_train_data[key].shape}") 

In [ ]:
pgl_test_data = np.load(f'/home/oarowolo/workfile/OPFData/data/PGLearn/nminusone_topology/{system_size}_ieee-nminus1_test.npz')

# Get all keys
print("Available keys in the dataset:")
for key in pgl_test_data.files:
    # Print the key and its array shape
    print(f"{key}: shape {pgl_test_data[key].shape}")

In [ ]:
def load_grid_data(grid_size:int):
   # Load the data
    data = np.load(f'/home/oarowolo/workfile/OPFData/data/OPFData/full_topology/{grid_size}bus_combined_dataset.npz')

    # Get all keys
    print("Available keys in the dataset:")
    for key in data.files:
        # Print the key and its array shape
        print(f"{key}: shape {data[key].shape}") 
        
        
        
        # Access grid input features
        grid_bus = data['grid_bus']  
        grid_generator = data['grid_generator']
        grid_load = data['grid_load']
        grid_shunt = data['grid_shunt']
        grid_ac_line_features = data['grid_ac_line_features']
        grid_transformer_features = data['grid_transformer_features']
        grid_ac_line_receivers = data['grid_ac_line_receivers']
        grid_ac_line_senders = data['grid_ac_line_senders']
        grid_transformer_senders = data['grid_transformer_senders']
        grid_transformer_receivers = data['grid_transformer_receivers']
        
        solution_bus = data['solution_bus']  
        solution_generator = data['solution_generator'] 
        solution_objective = data['metadata_objective']
        
        solution_objective = solution_objective.reshape(-1,1)
            
        grid_generator_link_receivers = data['grid_generator_link_receivers']
        grid_load_link_receivers = data['grid_load_link_receivers']
        grid_shunt_link_receivers = data['grid_shunt_link_receivers']
        
        generator_indices = grid_generator_link_receivers[0]
        load_indices = grid_load_link_receivers[0]
        shunt_indices = grid_shunt_link_receivers[0]
        grid_transformer_senders = grid_transformer_senders[0]
        grid_transformer_receivers = grid_transformer_receivers[0]
        grid_ac_line_senders  = grid_ac_line_senders[0]
        grid_ac_line_receivers = grid_ac_line_receivers[0]

        branch_list = list(zip(grid_ac_line_senders, grid_ac_line_receivers))
        transformer_list = list(zip(grid_transformer_senders, grid_transformer_receivers))
        for k in transformer_list:
            branch_list.append(k)
        
        return grid_bus, grid_generator, grid_load, grid_shunt, grid_ac_line_features,grid_ac_line_senders,grid_ac_line_receivers, grid_transformer_features, grid_transformer_senders, grid_transformer_receivers, solution_bus, solution_generator, solution_objective, generator_indices, load_indices, shunt_indices, branch_list

In [ ]:
grid_bus, grid_generator, grid_load, grid_shunt, grid_ac_line_features,grid_ac_line_senders,grid_ac_line_receivers, grid_transformer_features, grid_transformer_senders, grid_transformer_receivers,solution_bus, solution_generator, solution_objective, generator_indices,load_indices, shunt_indices,branch_list = load_grid_data(system_size)

In [ ]:
## We have to use the load inputs, gen and bus outputs and objective costs from the PGLearn data
## Keep the rest of the system features from OPFData since it's literally the same system
## We would use the rest of the system features to compute constraint satisfaction and so on.

In [ ]:
grid_load = pgl_train_data['grid_load']
solution_bus = pgl_train_data['solution_bus']  
solution_bus = solution_bus[:, :, ::-1]  # reverse the order of Va and Vm to match OPFData
solution_generator = pgl_train_data['solution_generator']
solution_objective = pgl_train_data['metadata_objective']

In [ ]:
train_gen_status = pgl_train_data['grid_generator']
train_branch_status = pgl_train_data['grid_branch']

In [ ]:
test_gen_status = pgl_test_data['grid_generator']
test_branch_status = pgl_test_data['grid_branch']

In [ ]:
default_branch_indices_14_bus = np.array([[ 0,  0,  1,  1,  1,  2,  3,  3,  3,  4,  5,  5,  5,  6,  6,  8,8,  9, 11, 12],
[ 1,  4,  2,  3,  4,  3,  4,  6,  8,  5, 10, 11, 12,  7,  8,  9, 13, 10, 12, 13]])

In [ ]:
default_branch_indices_30_bus = np.array([[ 0,  0,  1,  2,  1,  1,  3,  4,  5,  5,  5,  5,  8,  8,  3, 11,11, 11, 11, 13, 15, 14, 17, 18,  9,  9,  9,  9, 20, 14, 21, 22,23, 24, 24, 27, 26, 26, 28,  7,  5],
[ 1,  2,  3,  3,  4,  5,  5,  6,  6,  7,  8,  9, 10,  9, 11, 12,13, 14, 15, 14, 16, 17, 18, 19, 19, 16, 20, 21, 21, 22, 23, 23,24, 25, 26, 26, 28, 29, 29, 27, 27]])

In [ ]:
default_branch_indices_57_bus = np.array([[ 0,  1,  2,  3,  3,  5,  5,  7,  8,  8,  8,  8, 12, 12,  0,  0, 0,  2,  3,  3,  4,  6,  9, 10, 11, 11, 11, 13, 17, 18, 20, 20, 21, 22, 23, 23, 23, 25, 26, 27,  6, 24, 29, 30, 31, 33, 33, 34, 35, 36, 36, 35, 21, 10, 40, 40, 37, 14, 13, 45, 46, 47, 48, 49, 9, 12, 28, 51, 52, 53, 10, 43, 39, 55, 55, 38, 56, 37, 37,  8],
[ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 14, 15,16, 14, 17, 17,  5,  7, 11, 12, 12, 15, 16, 14, 18, 19, 19, 21,22, 23, 24, 24, 25, 26, 27, 28, 28, 29, 30, 31, 32, 31, 34, 35,36, 37, 38, 39, 37, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 50, 48, 51, 52, 53, 54, 42, 44, 55, 40, 41, 56, 55, 48, 47, 54]])

In [ ]:
default_branch_indices_118_bus = np.array([[  0,   0,   3,   2,   4,   5,   7,   7,   8,   3,   4,  10,   1, 2,   6,  10,  11,  12,  13,  11,  14,  15,  16,  17,  18,  14, 19,  20,  21,  22,  22,  25,  24,  26,  27,  29,   7,  25,  16, 28,  22,  30,  26,  14,  18,  34,  34,  32,  33,  33,  37,  36, 36,  29,  38,  39,  39,  40,  42,  33,  43,  44,  45,  45,  46,41,  41,  44,  47,  48,  48,  50,  51,  52,  48,  48,  53,  53, 54,  55,  49,  55,  50,  53,  55,  55,  54,  58,  58,  59,  59,60,  62,  62,  63,  37,  63,  48,  48,  61,  61,  64,  65,  64,46,  48,  67,  68,  23,  69,  23,  70,  70,  69,  69,  68,  73,75,  68,  74,  76,  77,  76,  76,  78,  67,  80,  76,  81,  82,82,  83,  84,  85,  84,  84,  87,  88,  88,  89,  88,  88,  90,91,  91,  92,  93,  79,  81,  93,  79,  79,  79,  91,  93,  94,95,  97,  98,  99,  91, 100,  99,  99, 102, 102,  99, 103, 104,104, 104, 105, 107, 102, 108, 109, 109,  16,  31,  31,  26, 113, 67,  11,  74,  75],
[  1,   2,   4,   4,   5,   6,   8,   4,   9,  10,  10,  11,  11,11,  11,  12,  13,  14,  14,  15,  16,  16,  17,  18,  19,  18,20,  21,  22,  23,  24,  24,  26,  27,  28,  16,  29,  29,  30,30,  31,  31,  31,  32,  33,  35,  36,  36,  35,  36,  36,  38,39,  37,  39,  40,  41,  41,  43,  42,  44,  45,  46,  47,  48,48,  48,  48,  48,  49,  50,  51,  52,  53,  53,  53,  54,  55,55,  56,  56,  57,  57,  58,  58,  58,  58,  59,  60,  60,  61,61,  58,  63,  60,  64,  64,  65,  65,  65,  66,  65,  66,  67,68,  68,  68,  69,  69,  70,  71,  71,  72,  73,  74,  74,  74,76,  76,  76,  77,  78,  79,  79,  79,  80,  79,  81,  82,  83,84,  84,  85,  86,  87,  88,  88,  89,  89,  90,  91,  91,  91,92,  93,  93,  94,  95,  95,  95,  96,  97,  98,  99,  99,  95,96,  99,  99, 100, 101, 101, 102, 103, 103, 104, 105, 104, 105,106, 107, 106, 108, 109, 109, 110, 111, 112, 112, 113, 114, 114,115, 116, 117, 117]])

In [ ]:
#just to separate the default transformer branches from the line branches
transformer_branches = branch_list[-grid_transformer_features.shape[1]:]
# Compare all edges at once
edge_list = default_branch_indices_14_bus ################ change this for each system!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! 
transformer_branches = np.array(transformer_branches)
matches = (edge_list[0][:, None] == transformer_branches[:, 0]) & (edge_list[1][:, None] == transformer_branches[:, 1])
transformer_indices = np.where(matches.any(axis=1))[0]
line_indices = np.where(matches.any(axis=1) == False)[0]

In [ ]:
line_indices

In [ ]:
transformer_indices

In [ ]:
grid_ac_line_senders = grid_ac_line_senders.reshape(-1,1)
grid_ac_line_receivers = grid_ac_line_receivers.reshape(-1,1)
grid_transformer_senders = grid_transformer_senders.reshape(-1,1)
grid_transformer_receivers = grid_transformer_receivers.reshape(-1,1)

In [ ]:
# ── Helper: build edge_inputs for an arbitrary subset of active branches ──────

## This is built with the same assumption the branch_list has, which is that only lines are built before transformers are added
def build_edge_inputs(ac_feats, xfmr_feats, active_ac_idx, active_xfmr_idx):
    """
    Rearranges ac_line and transformer features into the unified 11-column
    edge_inputs format, using only the active (status != 0) branches.

    Columns 0-8  : branch parameters (r, x, b, etc.)
    Column  9    : 1.0 for AC lines, 0.0 for transformers
    Column  10   : 0.0 for AC lines, 1.0 for transformers (set below)
    """
    n_active_ac   = len(active_ac_idx)
    n_active_xfmr = len(active_xfmr_idx)
    total         = n_active_ac + n_active_xfmr

    ei = np.zeros((total, 11))

    if n_active_ac > 0:
        ei[:n_active_ac, :9]   = ac_feats[active_ac_idx]
        ei[:n_active_ac, 9:10] = 1.0   # is_ac_line flag

    if n_active_xfmr > 0:
        xf = xfmr_feats[active_xfmr_idx]
        ei[n_active_ac:, :2]   = xf[:, :2]
        ei[n_active_ac:, 2:4]  = xf[:, 9:]
        ei[n_active_ac:, 4:9]  = xf[:, 2:7]
        ei[n_active_ac:, 9:]   = xf[:, 7:9]

    return ei

In [ ]:
def compute_gandb(edge_inputs):

    line_r = edge_inputs[:,4:5]
    line_x = edge_inputs[:,5:6]

    line_g = line_r/(line_r**2 + line_x**2)
    line_b = -line_x/(line_r**2 + line_x**2)

    return line_g, line_b


In [ ]:
# edge_g, edge_b = compute_gandb(edge_inputs)

In [ ]:
def get_B_matrix(N, edges, edge_weights):
    # Create a zero tensor of shape (N,N)
    B_matrix = torch.zeros((N, N)).to(torch.float64)
    
    # Unpack the edges into source and destination nodes
    sources, destinations = zip(*edges)
    
    # Use advanced indexing to place weights in the right spots
    B_matrix[sources, destinations] = edge_weights.squeeze()
    B_matrix[destinations, sources] = edge_weights.squeeze()
    return B_matrix

In [ ]:
def adjacency_to_laplacian(B_adj):

    # Ensure matrix is square
    assert B_adj.shape[0] == B_adj.shape[1], "Input must be square"

    # Copy to avoid modifying original
    B_laplacian = B_adj.copy()

    # Set diagonal as row sum of adjacency (i.e., degree)
    np.fill_diagonal(B_laplacian, -B_adj.sum(axis=1))

    return B_laplacian


In [ ]:
def another_effective_resistance_matrix(b_mat):
    """Compute effective resistance using a more efficient approach."""
    
    
    # Compute sparse Laplacian
    L = sp.csc_matrix(b_mat)
    num_nodes = L.shape[0]

    # Regularized Laplacian
    L_reg = L + 1e-10 * sp.eye(num_nodes)

    # Precompute factorization (Much faster than CG)
    L_solver = spla.factorized(L_reg)

    # Compute diagonal of pseudoinverse
    I = np.eye(num_nodes)
    L_plus_diag = np.array([L_solver(I[:, i])[i] for i in range(num_nodes)])

    # Compute resistance efficiently
    eff_res_matrix = np.zeros((num_nodes, num_nodes))

    for i in range(num_nodes):
        for j in range(i+1, num_nodes):
            e_ij = I[:, i] - I[:, j]
            x = L_solver(e_ij)  # Solve in one step

            resistance = np.dot(e_ij, x)
            eff_res_matrix[i, j] = resistance
            eff_res_matrix[j, i] = resistance  # Symmetric

    return eff_res_matrix, L_solver

In [ ]:
def effective_resistance_matrix(b_mat):
    """
    Computes the effective resistance matrix from a susceptance adjacency matrix.
    Converts it into a Laplacian first.
    
    Parameters:
    b_mat (ndarray): weighted laplacian matrix
    
    Returns:
    ndarray: Effective resistance matrix (N x N)
    """
    # Ensure symmetry
    L = b_mat

    n = L.shape[0]

    # Remove reference node (last row and column) to deal with singularity
    keep = np.arange(n - 1)
    L_reduced = L[np.ix_(keep, keep)]

    # Invert reduced Laplacian
    L_reduced_inv = np.linalg.inv(L_reduced)

    # Expand to full pseudoinverse
    L_plus = np.zeros((n, n))
    L_plus[np.ix_(keep, keep)] = L_reduced_inv

    # Project to orthogonal component (to make it true pseudoinverse)
    I = np.eye(n)
    ones = np.ones((n, n)) / n
    L_plus = (I - ones) @ L_plus @ (I - ones)

    # Compute resistance: R_ij = L^+_ii + L^+_jj - 2L^+_ij
    diag = np.diag(L_plus)
    R = diag[:, None] + diag[None, :] - 2 * L_plus
    return R

In [ ]:
# b_mat = np.array(B_weighted)

In [ ]:
# B_lap = adjacency_to_laplacian(b_mat)

In [ ]:
# e_R = effective_resistance_matrix(B_lap)

In [ ]:
# e_R_norm = e_R/e_R.max()

In [ ]:
# Vectorized version (more efficient for large matrices)
def compute_row_statistics_vectorized(resistance_matrix):
    """
    Vectorized computation of row statistics excluding diagonal elements.
    
    Parameters:
    resistance_matrix: numpy array of shape (N, N) with values between 0 and 1
    
    Returns:
    numpy array of shape (N, 5) with columns: [mean, median, std, max, min]
    """
    N = resistance_matrix.shape[0]
    
    # Create a mask to exclude diagonal elements
    mask = ~np.eye(N, dtype=bool)
    
    # Initialize result matrix
    stats_matrix = np.zeros((N, 5))
    
    # For each row, extract non-diagonal elements and compute statistics
    for i in range(N):
        row_no_diag = resistance_matrix[i, mask[i]]
        
        stats_matrix[i, 0] = np.mean(row_no_diag)
        stats_matrix[i, 1] = np.median(row_no_diag)
        stats_matrix[i, 2] = np.std(row_no_diag)
        stats_matrix[i, 3] = np.max(row_no_diag)
        stats_matrix[i, 4] = np.min(row_no_diag)
    
    return stats_matrix

In [ ]:
# raw_PE = compute_row_statistics_vectorized(e_R)

In [ ]:
# bus_pe = torch.tensor(raw_PE,dtype=torch.float)

In [ ]:
def compute_bus_pe(ac_feats, xfmr_feats, active_ac_idx, active_xfmr_idx,
                   ac_senders, ac_receivers, xfmr_senders, xfmr_receivers):
    """Compute the 14x5 bus positional encoding for one N-1 topology."""
    ei = build_edge_inputs(ac_feats, xfmr_feats, active_ac_idx, active_xfmr_idx)

    active_s = np.concatenate([ac_senders[active_ac_idx].flatten(),
                                xfmr_senders[active_xfmr_idx].flatten()])
    active_r = np.concatenate([ac_receivers[active_ac_idx].flatten(),
                                xfmr_receivers[active_xfmr_idx].flatten()])
    branch_list = list(zip(active_s, active_r))

    _, edge_b  = compute_gandb(ei)
    B_weighted = get_B_matrix(system_size, branch_list, torch.tensor(edge_b))
    B_lap      = adjacency_to_laplacian(np.array(B_weighted))
    e_R        = effective_resistance_matrix(B_lap)
    raw_PE     = compute_row_statistics_vectorized(e_R)
    return torch.tensor(raw_PE, dtype=torch.float)

In [ ]:
n_ac    = grid_ac_line_senders.shape[0]
n_xfmr  = grid_transformer_senders.shape[0]

In [ ]:
# Full-topology PE (used for generator N-1 contingencies)
full_active_ac   = np.arange(n_ac)
full_active_xfmr = np.arange(n_xfmr)
full_topology_pe = compute_bus_pe(
    grid_ac_line_features[0], grid_transformer_features[0],
    full_active_ac, full_active_xfmr,
    grid_ac_line_senders, grid_ac_line_receivers,
    grid_transformer_senders, grid_transformer_receivers
)

contingency_pe_cache = {}   # key: (tuple(active_ac_idx), tuple(active_xfmr_idx))

In [ ]:
def get_contingency_key(sample_idx,data):
    ac_status   = data[sample_idx, line_indices, :]
    xfmr_status = data[sample_idx, transformer_indices, :]
    active_ac   = tuple(np.where(ac_status   != 0)[0].tolist())
    active_xfmr = tuple(np.where(xfmr_status != 0)[0].tolist())
    return active_ac, active_xfmr

In [ ]:
def train_val_test_split(data_len, train_ratio=0.85, val_ratio=0.15, seed=42):

    # Set random seed for reproducibility
    torch.manual_seed(seed)

    # Shuffle indices
    num_samples = data_len
    indices = torch.randperm(num_samples)

    # Compute split sizes
    train_size = int(train_ratio * num_samples)
    val_size = int(val_ratio * num_samples)

    # Split indices
    train_indices = indices[:train_size]
    val_indices = indices[train_size:]
    

    return train_indices, val_indices

In [ ]:
data_len = grid_load.shape[0]

In [ ]:
train_indices, val_indices = train_val_test_split(data_len)

In [ ]:
N_train = grid_load.shape[0]
sample_pe_list = []

for i in range(N_train):
    active_ac, active_xfmr = get_contingency_key(i, train_branch_status)
    key = (active_ac, active_xfmr)
    
    if key not in contingency_pe_cache:
        active_ac_idx   = np.array(active_ac)
        active_xfmr_idx = np.array(active_xfmr)

        # If all branches are active, this is a generator contingency → reuse full PE
        if len(active_ac_idx) == n_ac and len(active_xfmr_idx) == n_xfmr:
            contingency_pe_cache[key] = full_topology_pe
        else:
            contingency_pe_cache[key] = compute_bus_pe(
                grid_ac_line_features[i], grid_transformer_features[i],
                active_ac_idx, active_xfmr_idx,
                grid_ac_line_senders, grid_ac_line_receivers,
                grid_transformer_senders, grid_transformer_receivers
            )

    sample_pe_list.append(contingency_pe_cache[key])

train_sample_pe = torch.stack(sample_pe_list)   # (N_train, 14, 5)
print(f"  Unique contingencies found: {len(contingency_pe_cache)}")
print(f"  sample_pe shape: {train_sample_pe.shape}")
# torch.save(train_sample_pe, f'PGLearn_train_{system_size}_N-1_PE_encoding.pth')

In [ ]:
batch_size = 256

In [ ]:
import torch
from torch_geometric.data import HeteroData

def create_grid_hetero_data(
    grid_bus,                    # (300000, 14, 4)
    grid_generator,              # (300000, 5, 11)
    grid_load,                   # (300000, 11, 2)
    grid_shunt,                  # (300000, 1, 2)
    grid_ac_line_features,       # (300000, 17, 9)
    grid_transformer_features,   # (300000, 3, 11)
    grid_ac_line_senders,        # (17, 1)
    grid_ac_line_receivers,      # (17, 1)
    grid_transformer_senders,    # (3, 1)
    grid_transformer_receivers,  # (3, 1)
    generator_indices,           # Indices connecting generators to buses
    load_indices,                # Indices connecting loads to buses
    shunt_indices,               # Indices connecting shunts to buses
    solution_bus,                # (300000, 14, 2) - solutions for bus nodes
    solution_generator,          # (300000, 5, 2) - solutions for generator nodes
    sample_bus_pe,
    gen_status,
    branch_status,
    batch_idx=0                  # Batch index to extract (or None for all batches)
):
    
    # Create a new HeteroData instance
    data = HeteroData()
    
    # Process a single batch if specified, otherwise we'd need to handle batching differently
    if batch_idx is not None:
        # Extract features for the specified batch
        bus_pe = torch.tensor(sample_bus_pe[batch_idx], dtype=torch.float)
        bus_features = torch.tensor(grid_bus[batch_idx], dtype=torch.float)
        generator_features = torch.tensor(grid_generator[batch_idx], dtype=torch.float)
        load_features = torch.tensor(grid_load[batch_idx], dtype=torch.float)
        shunt_features = torch.tensor(grid_shunt[batch_idx], dtype=torch.float)
        
        ac_line_features = torch.tensor(grid_ac_line_features[batch_idx], dtype=torch.float)
        transformer_features = torch.tensor(grid_transformer_features[batch_idx], dtype=torch.float)
        
        # Extract solution values for the specified batch
        bus_solutions = torch.tensor(solution_bus[batch_idx], dtype=torch.float)

        #########################################since, gen_features, line_feature, and xformer features are fixed except where they are absent, use indexing to select present components
        gen_active_mask  = gen_status[batch_idx,:,0] != 0.0  # (5,)  bool
        ac_active_mask   = branch_status[batch_idx,line_indices,0] != 0.0  # (17,) bool
        xfmr_active_mask = branch_status[batch_idx,transformer_indices,0] != 0.0  # (3,)  bool

        # ── Filter connectivity to match active elements ───────────────────────
        # generator_indices maps generator slots → bus ids; mask with same gen mask
        active_gen_indices   = generator_indices[gen_active_mask]         # (4 or 5,)
        active_ac_senders    = grid_ac_line_senders[ac_active_mask]       # (16 or 17, 1)
        active_ac_receivers  = grid_ac_line_receivers[ac_active_mask]
        active_xfmr_senders  = grid_transformer_senders[xfmr_active_mask]
        active_xfmr_receivers= grid_transformer_receivers[xfmr_active_mask]
        
        # gen solutions first
        generator_solutions = torch.tensor(solution_generator[batch_idx], dtype=torch.float)


        # Add node features
        data['bus'].x = bus_features
        data['generator'].x = generator_features[gen_active_mask]
        data['load'].x = load_features
        data['shunt'].x = shunt_features

        #Add node positional encoding
        data['bus'].pe = bus_pe
        data['generator'].pe = bus_pe[active_gen_indices] 
        data['load'].pe = bus_pe[load_indices] 
        data['shunt'].pe = bus_pe[shunt_indices] 
    
        
        # Add solution values as target values (y)
        data['bus'].y = bus_solutions
       
        data['generator'].y = generator_solutions[gen_active_mask]
        
        # Add edge indices and features for AC lines (bus to bus)
        senders = torch.tensor(active_ac_senders.flatten(), dtype=torch.long)
        receivers = torch.tensor(active_ac_receivers.flatten(), dtype=torch.long)
        edge_index = torch.stack([senders, receivers], dim=0)
        data['bus', 'ac_line', 'bus'].edge_index = edge_index
        data['bus', 'ac_line', 'bus'].edge_attr = ac_line_features[ac_active_mask]
        
        # Add edge indices and features for transformers (bus to bus)
        senders = torch.tensor(active_xfmr_senders.flatten(), dtype=torch.long)
        receivers = torch.tensor(active_xfmr_receivers.flatten(), dtype=torch.long)
        edge_index = torch.stack([senders, receivers], dim=0)
        data['bus', 'transformer', 'bus'].edge_index = edge_index
        data['bus', 'transformer', 'bus'].edge_attr = transformer_features[xfmr_active_mask]
        
        # Add pseudo-edges from generators to buses
        gen_to_bus = torch.tensor(active_gen_indices.flatten(), dtype=torch.long)
        gen_indices = torch.arange(len(gen_to_bus), dtype=torch.long)
        gen_edge_index = torch.stack([gen_indices, gen_to_bus], dim=0)
        data['generator', 'connects_to', 'bus'].edge_index = gen_edge_index
        data['generator', 'connects_to', 'bus'].edge_attr = torch.ones((len(gen_to_bus), 3))
        
        # Add pseudo-edges from loads to buses
        load_to_bus = torch.tensor(load_indices.flatten(), dtype=torch.long)
        load_indices = torch.arange(len(load_to_bus), dtype=torch.long)
        load_edge_index = torch.stack([load_indices, load_to_bus], dim=0)
        data['load', 'connects_to', 'bus'].edge_index = load_edge_index       
        data['load', 'connects_to', 'bus'].edge_attr = torch.ones((len(load_to_bus), 3))
        
        # Add pseudo-edges from shunts to buses
        shunt_to_bus = torch.tensor(shunt_indices.flatten(), dtype=torch.long)
        shunt_indices = torch.arange(len(shunt_to_bus), dtype=torch.long)
        shunt_edge_index = torch.stack([shunt_indices, shunt_to_bus], dim=0)
        data['shunt', 'connects_to', 'bus'].edge_index = shunt_edge_index
        data['shunt', 'connects_to', 'bus'].edge_attr = torch.ones((len(shunt_to_bus), 3))
    
    else:
        raise NotImplementedError("Processing all batches at once is not implemented in this example")
    
    return data



def create_dataloader(
    grid_bus,
    grid_generator,
    grid_load,
    grid_shunt,
    grid_ac_line_features,
    grid_transformer_features,
    grid_ac_line_senders,
    grid_ac_line_receivers,
    grid_transformer_senders,
    grid_transformer_receivers,
    generator_indices,
    load_indices,
    shunt_indices,
    solution_bus,
    solution_generator,
    sample_bus_pe,
    gen_status,
    branch_status,
    batch_size=batch_size,
    shuffle = True
):
    """
    Create a dataloader for the heterogeneous graph data with solutions.
    """
    from torch_geometric.loader import DataLoader
    
    # Create a list of HeteroData objects
    dataset = []
    
    for i in range(len(grid_bus)):  # Process up to 1000 samples for this example
        data = create_grid_hetero_data(
            grid_bus, 
            grid_generator,
            grid_load,
            grid_shunt,
            grid_ac_line_features,
            grid_transformer_features,
            grid_ac_line_senders,
            grid_ac_line_receivers,
            grid_transformer_senders,
            grid_transformer_receivers,
            generator_indices,
            load_indices,
            shunt_indices,
            solution_bus,
            solution_generator,
            sample_bus_pe,
            gen_status,
            branch_status,
            batch_idx=i
        )
        dataset.append(data)
    
    # Create a DataLoader
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,num_workers=min(8, torch.get_num_threads()),pin_memory=True,persistent_workers=True,prefetch_factor=2)
    
    return loader

In [ ]:
# for features we know are constant, we can simply choose the first index and multiply by the first dimension of PGLearn data 

In [ ]:
samp_grid_bus = grid_bus[0]
samp_grid_generator = grid_generator[0]
samp_grid_shunt = grid_shunt[0]
samp_grid_ac_line_features = grid_ac_line_features[0]
samp_grid_transformer_features = grid_transformer_features[0]

In [ ]:
grid_bus = np.repeat(samp_grid_bus[np.newaxis, :, :], grid_load.shape[0], axis=0)
grid_generator = np.repeat(samp_grid_generator[np.newaxis, :, :], grid_load.shape[0], axis=0)
grid_shunt = np.repeat(samp_grid_shunt[np.newaxis, :, :], grid_load.shape[0], axis=0)
grid_ac_line_features = np.repeat(samp_grid_ac_line_features[np.newaxis, :, :], grid_load.shape[0], axis=0)
grid_transformer_features = np.repeat(samp_grid_transformer_features[np.newaxis, :, :], grid_load.shape[0], axis=0)

In [ ]:
#create a separate test input and output

In [ ]:
test_grid_load = pgl_test_data['grid_load']
test_solution_bus = pgl_test_data['solution_bus']
test_solution_bus = test_solution_bus[:, :, ::-1]  # reverse the order of Va and Vm to match OPFData
test_solution_generator = pgl_test_data['solution_generator']
test_solution_objective = pgl_test_data['metadata_objective']

In [ ]:
test_grid_bus = np.repeat(samp_grid_bus[np.newaxis, :, :], test_grid_load.shape[0], axis=0)
test_grid_generator = np.repeat(samp_grid_generator[np.newaxis, :, :], test_grid_load.shape[0], axis=0)
test_grid_shunt = np.repeat(samp_grid_shunt[np.newaxis, :, :], test_grid_load.shape[0], axis=0)
test_grid_ac_line_features = np.repeat(samp_grid_ac_line_features[np.newaxis, :, :], test_grid_load.shape[0], axis=0)
test_grid_transformer_features = np.repeat(samp_grid_transformer_features[np.newaxis, :, :], test_grid_load.shape[0], axis=0)

In [ ]:
test_indices = torch.arange(test_grid_load.shape[0])

In [ ]:
# Build per-sample bus_pe for test set (reuses cache from training)
N_test = test_grid_load.shape[0]
test_sample_pe_list = []
for i in range(N_test):
    ac_status   = test_branch_status[i,line_indices,0]
    xfmr_status = test_branch_status[i,transformer_indices,0]
    active_ac   = tuple(np.where(ac_status   != 0)[0].tolist())
    active_xfmr = tuple(np.where(xfmr_status != 0)[0].tolist())
    key = (active_ac, active_xfmr)

    if key not in contingency_pe_cache:
        # Unseen contingency (e.g. test has a contingency not in train)
        active_ac_idx   = np.array(active_ac)
        active_xfmr_idx = np.array(active_xfmr)
        if len(active_ac_idx) == n_ac and len(active_xfmr_idx) == n_xfmr:
            contingency_pe_cache[key] = full_topology_pe
        else:
            contingency_pe_cache[key] = compute_bus_pe(
                test_grid_ac_line_features[i], test_grid_transformer_features[i],
                active_ac_idx, active_xfmr_idx,
                grid_ac_line_senders, grid_ac_line_receivers,
                grid_transformer_senders, grid_transformer_receivers
            )

    test_sample_pe_list.append(contingency_pe_cache[key])

test_sample_pe = torch.stack(test_sample_pe_list)   # (N_train, 14, 5)
print(f"  Unique contingencies found: {len(contingency_pe_cache)}")
print(f"  sample_pe shape: {test_sample_pe.shape}")
torch.save(test_sample_pe, f'PGLearn_test_{system_size}_N-1_PE_encoding.pth')

In [ ]:
train_loader = create_dataloader(grid_bus[train_indices],
                                grid_generator[train_indices],
                                grid_load[train_indices],
                                grid_shunt[train_indices],
                                grid_ac_line_features[train_indices],
                                grid_transformer_features[train_indices],
                                grid_ac_line_senders,
                                grid_ac_line_receivers,
                                grid_transformer_senders,
                                grid_transformer_receivers,
                                generator_indices,
                                load_indices,
                                shunt_indices,
                                solution_bus[train_indices],
                                solution_generator[train_indices],
                                train_sample_pe[train_indices],
                                train_gen_status[train_indices],
                                train_branch_status[train_indices], 
                                batch_size=batch_size,
                                shuffle = True)

In [ ]:
val_loader = create_dataloader(grid_bus[val_indices],
                                grid_generator[val_indices],
                                grid_load[val_indices],
                                grid_shunt[val_indices],
                                grid_ac_line_features[val_indices],
                                grid_transformer_features[val_indices],
                                grid_ac_line_senders,
                                grid_ac_line_receivers,
                                grid_transformer_senders,
                                grid_transformer_receivers,
                                generator_indices,
                                load_indices,
                                shunt_indices,
                                solution_bus[val_indices],
                                solution_generator[val_indices],
                                train_sample_pe[val_indices],
                                train_gen_status[val_indices],
                                train_branch_status[val_indices],
                                batch_size=batch_size,
                                shuffle = False)

In [ ]:
test_loader = create_dataloader(test_grid_bus[test_indices],
                                test_grid_generator[test_indices],
                                test_grid_load[test_indices],
                                test_grid_shunt[test_indices],
                                test_grid_ac_line_features[test_indices],
                                test_grid_transformer_features[test_indices],
                                grid_ac_line_senders,
                                grid_ac_line_receivers,
                                grid_transformer_senders,
                                grid_transformer_receivers,
                                generator_indices,
                                load_indices,
                                shunt_indices,
                                test_solution_bus[test_indices],
                                test_solution_generator[test_indices],
                                test_sample_pe[test_indices],
                                test_gen_status[test_indices],
                                test_branch_status[test_indices],
                                batch_size=batch_size,
                                shuffle = False)

In [ ]:
class MLP(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size, layers, layernorm=True, use_leaky=False): 
        super().__init__()
        # Use Sequential instead of ModuleList for faster forward pass
        modules = []
        for i in range(layers):
            modules.append(torch.nn.Linear(
                input_size if i == 0 else hidden_size,
                output_size if i == layers - 1 else hidden_size,
            ))
            if i != layers - 1:
                modules.append(torch.nn.ReLU())
            if use_leaky:
                modules.append(torch.nn.LeakyReLU(negative_slope=0.02))
        if layernorm:
            modules.append(torch.nn.LayerNorm(output_size))
        
        self.network = torch.nn.Sequential(*modules)
        self.reset_parameters()

    
    def reset_parameters(self): ## This is recently removed, weight initialisation seems to be doing more harm than good.
        for layer in self.network:
            if isinstance(layer, torch.nn.Linear):  
                layer.weight.data.normal_(0, 1 / math.sqrt(layer.in_features))
                layer.bias.data.fill_(0)
                
    def forward(self, x):
        # Sequential is faster than iterating through ModuleList
        return self.network(x)

In [ ]:
# from performer_pytorch import SelfAttention
from torch_geometric.utils import to_dense_batch
from torch_geometric.nn.attention import PerformerAttention

class HeteroPerformerLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads=1, dropout=0.0):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.attn = PerformerAttention(
            channels=hidden_dim,
            heads=num_heads
        )
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.norm3 = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

        # Post-attention MLP
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
    def forward(self, x_dict, xm_dict, batch_dict):
            # Flatten all node types
            flat_x, flat_xm, flat_batch = [], [], []
            slices = {}
            offset = 0
    
            for ntype in x_dict:
                x = x_dict[ntype]
                xm = xm_dict[ntype]
                b = batch_dict[ntype]  # batch indices for each node
    
                slices[ntype] = slice(offset, offset + x.size(0))
                flat_x.append(x)
                flat_xm.append(xm)
                flat_batch.append(b)
                offset += x.size(0)
    
            x_all = torch.cat(flat_x, dim=0)         # [N, D]
            xm_all = torch.cat(flat_xm, dim=0)       # [N, D]
            global_batch_all = torch.cat(flat_batch, dim=0) # [N]

            _, batch_all = torch.unique(global_batch_all, return_inverse=True)
            sorted_indices = torch.argsort(batch_all) # resort batch index to be ascending but that means I have to sort the xs myself too
            batch_all_sorted = batch_all[sorted_indices]
            x_all_sorted = x_all[sorted_indices]
            xm_all_sorted = xm_all[sorted_indices]
            # Convert to [B, N_max, D] and mask            
            x_dense, mask = to_dense_batch(x_all_sorted, batch_all_sorted)   # [B, N, D], [B, N]
            
            # Apply masked Performer attention
            x_attn = self.attn(x_dense, mask=mask)              # [B, N, D]
            # Residual + Norm
            xt_out = self.norm1(x_dense + x_attn)          
            # Unpad: [real_nodes, D]
            xt_out = xt_out[mask]
    
            x_comb = self.norm2(xt_out + xm_all_sorted)
            x_final = self.mlp(x_comb)

            # dont forget to give x_final it unsorted arrangement
            unsorted_x = torch.empty_like(x_final)
            unsorted_x[sorted_indices] = x_final
            x_final = unsorted_x
            # x_final = x_final[sorted_indices]

            # Unflatten by slice
            return {ntype: x_final[slices[ntype]] for ntype in x_dict.keys()}


In [ ]:
class HeteroInteractionNetwork(nn.Module):
    def __init__(self, node_types, edge_types,physical_edge_types, hidden_size, layers):
        super().__init__()
        
        self.physical_edge_types = physical_edge_types
        self.edge_updaters = nn.ModuleDict()
        for src, rel, dst in edge_types:
            edge_key = f"{src}_{rel}_{dst}"
            edge_type = (src, rel, dst)
            
            # For physical edges (ac_line and transformer), use node + edge features
            if edge_type in physical_edge_types:
                self.edge_updaters[edge_key] = MLP(hidden_size * 3, hidden_size, hidden_size, layers)
            else:
                # For other edge types, only use node features
                self.edge_updaters[edge_key] = MLP(hidden_size * 2, hidden_size, hidden_size, layers)
        
        # Create a node updater for each node type
        self.node_updaters = nn.ModuleDict({
            node_type: MLP(hidden_size * 2, hidden_size, hidden_size, layers)
            for node_type in node_types
        })

    
    def forward(self, x_dict, edge_indices_dict, edge_features_dict):
        # Store updated node and edge features
        updated_edge_features = {}
        
        # Prepare aggregated messages storage
        aggregated_messages = {node_type: torch.zeros_like(feat) 
                              for node_type, feat in x_dict.items()}
        
        # Process each edge type in parallel
        for edge_type, edge_index in edge_indices_dict.items():
            src_type, rel_type, dst_type = edge_type
            edge_key = f"{src_type}_{rel_type}_{dst_type}"
            
            # Get node features for this edge
            src, dst = edge_index
            x_i = x_dict[dst_type][dst]  # Destination nodes
            x_j = x_dict[src_type][src]  # Source nodes
            # Update edge features based on edge type
            if edge_type in self.physical_edge_types:
                # For physical edges, include edge features in message
                edge_feature = edge_features_dict[edge_type]
                edge_msg = torch.cat((x_i, x_j, edge_feature), dim=-1)
                updated_edge = self.edge_updaters[edge_key](edge_msg) ## this implementation may actually be buggy here, we are using only the new update in message aggreagation, we are actually discarding old messages
                ###########here is a new line where we take care of the potentially buggy residual
                updated_edge = updated_edge + edge_feature
                updated_edge_features[edge_type] =    updated_edge ##+ edge_feature   removed edge feature after doing the new line residual
            else:
                # For non-physical edges, only use node features
                edge_msg = torch.cat((x_i, x_j), dim=-1)
                updated_edge = self.edge_updaters[edge_key](edge_msg)
                
                # Check if we have existing edge features from previous layers
                # if edge_type in edge_features_dict:
                #     edge_feature = edge_features_dict[edge_type]
                #     updated_edge_features[edge_type] =  updated_edge ##+ edge_feature  ### removed residual connection for non-physical edges here
                # else:
                    #First layer - initialize with the computed edge features
                updated_edge_features[edge_type] = updated_edge #better to remove residual connection here because there were no edge features to start with
            
            # Add these 2 lines before scatter_add if we are using mixed-precision training
            aggregated_messages[dst_type] = aggregated_messages[dst_type].to(updated_edge.dtype)
            aggregated_messages[src_type] = aggregated_messages[src_type].to(updated_edge.dtype)
            # Efficient message aggregation using torch_scatter
            aggregated_messages[dst_type] = torch_scatter.scatter_add(updated_edge, dst, dim=0, out=aggregated_messages[dst_type])
            aggregated_messages[src_type] = torch_scatter.scatter_add(updated_edge, src, dim=0, out=aggregated_messages[src_type])
        
        # Update node features
        updated_nodes = {}
        for node_type, x in x_dict.items():
            # Combine node features with aggregated messages
            node_input = torch.cat((x, aggregated_messages[node_type]), dim=-1)
            node_update = self.node_updaters[node_type](node_input)
            updated_nodes[node_type] = x + node_update  # Residual connection # try without residual x + node_update
        
        return updated_nodes, updated_edge_features


In [ ]:
class HeteroInteractGNN(torch.nn.Module):
    def __init__(
        self,
        hidden_size=256,
        n_mp_layers=5,
        bus_features=4,
        gen_features=11,
        load_features=2,
        shunt_features=2,
        ac_line_features=9,
        transformer_features=11,
        connects_to_features=3,
        output_dim=2
    ):
        super().__init__()
        
        # Define node and edge types
        self.node_types = ['bus', 'generator', 'load', 'shunt']
        self.edge_types = [
            ('bus', 'ac_line', 'bus'),
            ('bus', 'transformer', 'bus'),
            ('generator', 'connects_to', 'bus'), # changed names of connectors here as well
            ('load', 'connects_to', 'bus'),
            ('shunt', 'connects_to', 'bus')
        ]

        self.physical_edge_types = [
            ('bus', 'ac_line', 'bus'),
            ('bus', 'transformer', 'bus')
        ]        
        # Node encoders - separate MLP for each node type
        self.node_encoders = nn.ModuleDict({
            'bus': MLP(bus_features, hidden_size, hidden_size-5, 2),
            'generator': MLP(gen_features, hidden_size, hidden_size-5, 2),
            'load': MLP(load_features, hidden_size, hidden_size-5, 2),
            'shunt': MLP(shunt_features, hidden_size, hidden_size-5, 2)
        })
        # self.node_encoders = nn.ModuleDict({
        #     'bus': nn.Linear(bus_features, hidden_size-5),
        #     'generator': nn.Linear(gen_features, hidden_size-5),
        #     'load': nn.Linear(load_features, hidden_size-5),
        #     'shunt': nn.Linear(shunt_features, hidden_size-5)
        # })

        self.global_attn_layers = nn.ModuleList([
            HeteroPerformerLayer(hidden_size)
            for _ in range(n_mp_layers)
        ])

        # Edge encoders - separate MLP for each edge type
        self.edge_encoders = nn.ModuleDict({
            'ac_line': MLP(ac_line_features, hidden_size, hidden_size, 2),
            'transformer': MLP(transformer_features, hidden_size, hidden_size, 2)
        })
        
        # self.edge_encoders = nn.ModuleDict({
        #     'ac_line': nn.Linear(ac_line_features, hidden_size),
        #     'transformer': nn.Linear(transformer_features, hidden_size)
        # })  
        
        # Interaction network layers
        self.n_mp_layers = n_mp_layers
        self.layers = torch.nn.ModuleList([
            HeteroInteractionNetwork(self.node_types, self.edge_types,self.physical_edge_types, hidden_size, 2)
            for _ in range(n_mp_layers)
        ])

        
        # Node decoders - separate for bus and generator
        self.node_decoders = nn.ModuleDict({
            'bus': MLP(hidden_size, hidden_size, output_dim, 2, layernorm=False),
            'generator': MLP(hidden_size, hidden_size, output_dim, 2,layernorm=False)
        })


    def forward(self, data):
        # Encode node features
        x_dict_init = {}
        x_dict = {}
        
        # Batch node encoding
        for node_type in self.node_types:
            if hasattr(data[node_type], 'x'):
                x_dict_init[node_type] = self.node_encoders[node_type](data[node_type].x)
                x_dict[node_type] = torch.cat([x_dict_init[node_type], data[node_type].pe], dim=-1)

        # Encode edge features
        edge_feature_dict = {}
        for src, rel, dst in self.edge_types:
            edge_type = (src, rel, dst)
            if (edge_type in self.physical_edge_types and 
                edge_type in data.edge_types and 
                hasattr(data[edge_type], 'edge_attr')):
                edge_feature_dict[edge_type] = self.edge_encoders[rel](data[edge_type].edge_attr)
        
        # Extract edge indices
        edge_index_dict = {
            edge_type: data[edge_type].edge_index
            for edge_type in self.edge_types
            if edge_type in data.edge_types and hasattr(data[edge_type], 'edge_index')
        }

        batch_dict = {
            ntype: data[ntype].batch
            for ntype in x_dict
        }
        
        # Apply message passing layers
        for i in range(self.n_mp_layers):
            xm_dict, edge_feature_dict = self.layers[i](x_dict, edge_index_dict, edge_feature_dict)
            x_dict = self.global_attn_layers[i](x_dict, xm_dict, batch_dict)

        # Apply decoders
        output = {}
        output['bus'] = torch.sigmoid(self.node_decoders['bus'](x_dict['bus']))
        output['generator'] = torch.sigmoid(self.node_decoders['generator'](x_dict['generator']))
        
        return output

In [ ]:
## wandb set-up
api_key = 'my_key'
wandb.login(key=api_key)

In [ ]:
def convert_voltage_bounds(model_input):

    num_nodes = model_input.shape[0]

    vmin = model_input[:,2:3]

    vmax = model_input[:,3:4]

    thetamin =  torch.tensor([-2.00]).to(device)
    thetamin = thetamin.tile((num_nodes,1))

    thetamax =  torch.tensor([2.00]).to(device)
    thetamax = thetamax.tile((num_nodes,1))

    bounds_up = torch.concat((thetamax, vmax), dim=1)
    bounds_down = torch.concat((thetamin, vmin),dim=1)


    return bounds_up, bounds_down

In [ ]:
def convert_power_bounds(model_input):

    num_nodes = model_input.shape[0]

    pmin = model_input[:,2:3]

    pmax = model_input[:,3:4]

    
    qmin = model_input[:,5:6]

    qmax = model_input[:,6:7]

    bounds_up = torch.concat((pmax, qmax), dim=1)
    bounds_down = torch.concat((pmin, qmin),dim=1)


    return bounds_up, bounds_down
    

In [ ]:
# Example training step
def train_model(model, trainloader, optimizer):
    
    model.train()
    total_loss = 0
    criterion = nn.MSELoss()
    # amp_dtype =  torch.bfloat16
    # scaler  = GradScaler(enabled=(amp_dtype == torch.bfloat16))

#     for batch in tqdm(trainloader, desc="Training"):
    for batch in trainloader:
    
        optimizer.zero_grad(set_to_none=True)
        
        batch = batch.to(device, non_blocking=True)
        # Get batch size from the batch tensor
        # batch_size = batch.batch.max().item() + 1

        # Forward pass
        # model.redraw_projection.redraw_projections()
        with autocast(device_type=str(device),dtype=torch.bfloat16):
            pred_dict = model(batch)
    
            voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
            voltage_up = voltage_up.to(device)
            voltage_down = voltage_down.to(device)
            voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
            voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)
    
            power_up, power_down = convert_power_bounds(batch['generator'].x)
            power_up = power_up.to(device)
            power_down = power_down.to(device)
            powers = pred_dict['generator'] * (power_up - power_down) + power_down
            powers = torch.clamp(powers,min=power_down, max=power_up)
            
    
            # voltage_loss = criterion(voltages, batch['bus'].y)
            # power_loss = criterion(powers, batch['generator'].y)
            # loss = voltage_loss + power_loss
            combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
            combined_outputs = torch.cat([voltages, powers], dim=0)
            loss = criterion(combined_targets, combined_outputs)
    
            # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()   
            
        total_loss += loss.item()
        
    return total_loss / len(trainloader)

In [ ]:
# Example training step
def validate_model(model, val_loader):
    
    model.eval()
    total_loss = 0
    criterion = nn.MSELoss()
    # amp_dtype =  torch.bfloat16
    # scaler  = GradScaler(enabled=(amp_dtype == torch.bfloat16))
    
#     for batch in tqdm(trainloader, desc="Training"):
    for batch in val_loader:
        
        batch = batch.to(device, non_blocking=True)
        # Get batch size from the batch tensor
        # batch_size = batch.batch.max().item() + 1

        # Forward pass
        with autocast(device_type=str(device),dtype=torch.bfloat16):
            pred_dict = model(batch)
    
            voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
            voltage_up = voltage_up.to(device)
            voltage_down = voltage_down.to(device)
            voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
            voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)
    
            power_up, power_down = convert_power_bounds(batch['generator'].x)
            power_up = power_up.to(device)
            power_down = power_down.to(device)
            powers = pred_dict['generator'] * (power_up - power_down) + power_down
            powers = torch.clamp(powers,min=power_down, max=power_up)
    
            combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
            combined_outputs = torch.cat([voltages, powers], dim=0)
    
            loss = criterion(combined_targets, combined_outputs)
     
        total_loss += loss.item()
        
    return total_loss / len(val_loader)

In [ ]:
@torch.no_grad()
def test_model(model, testloader):

    model.eval()
    criterion = nn.MSELoss()
    # amp_dtype =  torch.bfloat16
    # scaler  = GradScaler(enabled=(amp_dtype == torch.bfloat16))
    
    total_loss = 0.0
    voltage_predictions = []
    voltage_targets = []
    power_predictions = []
    power_targets = []
    
    for batch in testloader:

        batch = batch.to(device, non_blocking=True)
        # Get batch size from the batch tensor
        # batch_size = batch.batch.max().item() + 1

        # Forward pass
        with autocast(device_type=str(device),dtype=torch.bfloat16):
            pred_dict = model(batch)
    
            voltage_up, voltage_down = convert_voltage_bounds(batch['bus'].x)
            voltage_up = voltage_up.to(device)
            voltage_down = voltage_down.to(device)
            voltages = pred_dict['bus'] * (voltage_up - voltage_down) + voltage_down
            voltages = torch.clamp(voltages,min=voltage_down, max=voltage_up)
    
            power_up, power_down = convert_power_bounds(batch['generator'].x)
            power_up = power_up.to(device)
            power_down = power_down.to(device)
            powers = pred_dict['generator'] * (power_up - power_down) + power_down
            powers = torch.clamp(powers,min=power_down, max=power_up)
            
            combined_targets = torch.cat([batch['bus'].y, batch['generator'].y], dim=0)
            combined_outputs = torch.cat([voltages, powers], dim=0)
            loss = criterion(combined_targets, combined_outputs)
        

        total_loss += loss.item()

        # Store predictions and targets for overall metrics
        voltage_predictions.append(voltages.cpu())
        voltage_targets.append(batch['bus'].y.cpu())

        power_predictions.append(powers.cpu())
        power_targets.append(batch['generator'].y.cpu())

    
    return total_loss / len(testloader), voltage_predictions, voltage_targets, power_predictions, power_targets

In [ ]:
run = wandb.init(
      # Set the project where this run will be logged
      project="Towards_Generalization_of_GNN_for_ACOPF",
      # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
      name=f"PGLearn_{system_size}_N-1_HybridHeteroGNN_5_256_PQVT",
      # Track hyperparameters and run metadata
      config={
      "architecture": "GNN",
      "dataset": "PGLearn",
      "epochs": 100,
      })

In [ ]:
model = HeteroInteractGNN().to(device)

In [ ]:
tot_params = 0
for parameter in model.parameters():
  layer_ws = 1
  for val in parameter.shape:
      layer_ws*=val
  tot_params += layer_ws
print(f"Total number of parameters = {tot_params}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, fused=True, weight_decay=5e-8)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

In [ ]:
training_losses = []
validation_losses = []
best_valid_loss = float('inf')
early_stop_thresh = 100
best_epoch = -1
best_model_state = None
num_epochs = 100
for epoch in tqdm(range(num_epochs), desc="Training Progress"):
    train_loss = train_model(model, train_loader, optimizer)
    valid_loss = validate_model(model, val_loader)
    training_losses.append(train_loss)
    validation_losses.append(valid_loss)

    wandb.log({"training_loss": train_loss, "validation_loss": valid_loss})

    scheduler.step(train_loss)

    if epoch % 10 == 0:
      print(f'Epoch: {epoch}')
      print(f'\tTrain Loss: {train_loss:.4f}')
      print(f'\t Val. Loss: {valid_loss:.4f}')
    if valid_loss < best_valid_loss:
      best_valid_loss = valid_loss
      best_model_state = deepcopy(model.state_dict())

plt.subplots(figsize=(5,3))
plt.plot([i for i in range(len(training_losses))], training_losses, 'r', label='Training loss')
plt.plot([i for i in range(len(validation_losses))], validation_losses, 'g', label='Validation loss')
plt.legend()
plt.title(f'GNN Training and Validation loss',fontsize = 15)
plt.xlabel('Epochs',fontsize = 12)
plt.ylabel('MSE Loss',fontsize = 12)
plt.semilogy()

training_losses=np.array(training_losses)
validation_losses=np.array(validation_losses)

model.load_state_dict(best_model_state)
model.eval()

In [ ]:
torch.save(model.state_dict(), f"{system_size}_bus_N-1_HybridHeteroGNN_5_256_PQVT_PGLearn.pth")
wandb.save(f"{system_size}_bus_N-1_HybridHeteroGNN_5_256_PQVT_PGLearn.pth")  # Upload to WandB

In [ ]:
test_loss, v_predictions, v_targets,p_predictions, p_targets = test_model(model, test_loader)

In [ ]:
print('loss on test data is ', test_loss)

In [ ]:
v_predictions = torch.cat(v_predictions, dim=0)
v_targets = torch.cat(v_targets, dim=0)

In [ ]:
p_predictions = torch.cat(p_predictions, dim=0)
p_targets = torch.cat(p_targets, dim=0)

In [ ]:
v_predictions = v_predictions.reshape(-1,system_size, 2)
v_targets = v_targets.reshape(-1, system_size, 2)

In [ ]:
# p_predictions = p_predictions.reshape(-1,generator_indices.shape[0], 2)
# p_targets = p_targets.reshape(-1, generator_indices.shape[0], 2)

In [ ]:
# compute voltage magnitude loss  and voltage angle loss separately
calc_loss = nn.MSELoss()
voltage_angle_loss = calc_loss(v_predictions[:,:,0],v_targets[:,:,0])
voltage_magnitude_loss = calc_loss(v_predictions[:,:,1],v_targets[:,:,1])

In [ ]:
print('average voltage angle discrepancy is  ', voltage_angle_loss)
print('average voltage magnitude discrepancy is  ', voltage_magnitude_loss)

In [ ]:
calc_loss = nn.MSELoss()
active_power_loss = calc_loss(p_predictions[:,0],p_targets[:,0])
reactive_power_loss = calc_loss(p_predictions[:,1],p_targets[:,1])
print('average gen active power error is  ', active_power_loss)
print('average gen reactive power error is  ', reactive_power_loss)

In [ ]:
num_nodes = test_gen_status.squeeze(-1).sum(axis=1).astype(int).tolist()

In [ ]:
#Iterate over num_nodes to extract each graph's predictions
separate_p_predictions = []
index = 0  # To track position in concatenated predictions

for nodes in num_nodes:
    separate_p_predictions.append(p_predictions[index:index + nodes])  # Extract corresponding predictions
    index += nodes  # Move index forward


In [ ]:
#Iterate over num_nodes to extract each graph's predictions
separate_p_targets = []
index = 0  # To track position in concatenated predictions

for nodes in num_nodes:
    separate_p_targets.append(p_targets[index:index + nodes])  # Extract corresponding predictions
    index += nodes  # Move index forward

In [ ]:
load_demand = torch.zeros(test_grid_load[test_indices].shape[0], system_size, test_grid_load[test_indices].shape[-1])
load_demand[:,load_indices,:] = torch.tensor(test_grid_load[test_indices])

In [ ]:
test_branch_list = []

for k in range(len(test_branch_status)):
    ac_active_mask   = test_branch_status[k,line_indices,0] != 0.0  # (17,) bool
    xfmr_active_mask = test_branch_status[k,transformer_indices,0] != 0.0  # (3,)  bool
    active_ac_senders    = grid_ac_line_senders[ac_active_mask]       # (16 or 17, 1)
    active_ac_receivers  = grid_ac_line_receivers[ac_active_mask]
    active_xfmr_senders  = grid_transformer_senders[xfmr_active_mask]
    active_xfmr_receivers= grid_transformer_receivers[xfmr_active_mask]
    ac_senders = torch.tensor(active_ac_senders.flatten(), dtype=torch.long)
    ac_receivers = torch.tensor(active_ac_receivers.flatten(), dtype=torch.long)
    xfmr_senders = torch.tensor(active_xfmr_senders.flatten(), dtype=torch.long)
    xfmr_receivers = torch.tensor(active_xfmr_receivers.flatten(), dtype=torch.long)
    init_branch_list = list(zip(ac_senders, ac_receivers))
    transformer_list = list(zip(xfmr_senders, xfmr_receivers))
    for q in transformer_list:
        init_branch_list.append(q)
    test_branch_list.append(init_branch_list)
    
    


In [ ]:
def compute_homogenous_edges(branch_list,grid_ac_line_features,grid_transformer_features):

    edge_inputs = np.zeros((len(branch_list),11))
    
    edge_inputs[:grid_ac_line_features.shape[0],:9] = grid_ac_line_features  # rearranging edge inputs to align for transformers and transmission lines
    edge_inputs[grid_ac_line_features.shape[0]:,:2] =  grid_transformer_features[:,:2]
    edge_inputs[grid_ac_line_features.shape[0]:,2:4] =  grid_transformer_features[:,9:]
    edge_inputs[grid_ac_line_features.shape[0]:,4:9] =  grid_transformer_features[:,2:7]
    edge_inputs[grid_ac_line_features.shape[0]:,9:] =  grid_transformer_features[:,7:9]
    edge_inputs[:grid_ac_line_features.shape[0],9:10] = 1.0
    
    return torch.tensor(edge_inputs)

In [ ]:
def compute_gandb(edge_inputs):

    line_r = edge_inputs[:,4:5]
    line_x = edge_inputs[:,5:6]

    line_g = line_r/(line_r**2 + line_x**2)
    line_b = -line_x/(line_r**2 + line_x**2)

    return line_g, line_b


In [ ]:
# edge_g, edge_b = compute_gandb(edge_inputs)

In [ ]:
# edge_inputs = torch.tensor(edge_inputs)

In [ ]:
# def calculate_only_branch_flows(
#     demand: torch.Tensor,  # shape (batch_size,14,2) [real, imag]
#     voltage: torch.Tensor,  # shape (batch_size,14,2) [real, imag]
#     branches: list,  # list of 20 tuples (from_bus, to_bus)
#     Yks: torch.Tensor,  # shape (14,2) [real, imag] shunt admittance
#     Yij: torch.Tensor,  # shape (20,2) [real, imag] branch admittance
#     Yijc: torch.Tensor,  # shape (20,2) [real, imag] branch charging admittance
#     Tij: torch.Tensor,  # shape (20,2) [real, imag] transformation ratio
# ) -> torch.Tensor:
#     batch_size = demand.shape[0]
#     num_nodes = voltage.shape[1]
    
#     # Helper function for batched complex multiplication
#     def complex_mult_batch(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
#         return torch.stack([
#             a[..., 0] * b[..., 0] - a[..., 1] * b[..., 1],
#             a[..., 0] * b[..., 1] + a[..., 1] * b[..., 0]
#         ], dim=-1)

#     # Helper function for batched complex conjugate
#     def complex_conj_batch(x: torch.Tensor) -> torch.Tensor:
#         return torch.stack([x[..., 0], -x[..., 1]], dim=-1)

#     # Helper function for batched complex division
#     def complex_div_batch(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
#         denominator = b[..., 0]**2 + b[..., 1]**2
#         return torch.stack([
#             (a[..., 0] * b[..., 0] + a[..., 1] * b[..., 1]) / denominator,
#             (a[..., 1] * b[..., 0] - a[..., 0] * b[..., 1]) / denominator
#         ], dim=-1)

#     # Initialize generator power tensor
#     generator_power = torch.zeros_like(demand)
    
#     # Calculate shunt power terms for each node (vectorized)
#     v_mag_sq = torch.sum(voltage**2, dim=-1, keepdim=True)  # shape: (batch_size, 14, 1)
#     v_mag_sq = torch.cat([v_mag_sq, torch.zeros_like(v_mag_sq)], dim=-1)  # shape: (batch_size, 14, 2)
    
#     # Broadcast Yks to match batch dimension
#     Yks_batch = Yks.expand(batch_size, -1, -1)  # shape: (batch_size, 14, 2)

#     shunt_power = complex_mult_batch(complex_conj_batch(Yks_batch), v_mag_sq)
    
#     # Pre-allocate branch flows dictionary with tensors
#     branch_flows = {}
    
#     # Calculate branch flows (vectorized)
#     for idx, (i, j) in enumerate(branches):
        
#         # Get complex voltage at both ends
#         vi = voltage[:, i]  # shape: (batch_size, 2)
#         vj = voltage[:, j]  # shape: (batch_size, 2)
        
#         # First term calculations
#         vi_mag_sq = torch.sum(vi**2, dim=-1, keepdim=True)  # shape: (batch_size, 1)
#         vi_mag_sq = torch.cat([vi_mag_sq, torch.zeros_like(vi_mag_sq)], dim=-1)  # shape: (batch_size, 2)
        
#         tij_mag_sq = torch.sum(Tij[idx]**2).unsqueeze(0)
#         tij_mag_sq_tensor = torch.tensor([tij_mag_sq, 0.0], dtype=torch.float32).expand(batch_size, -1)
        
#         vi_over_tij_sq = complex_div_batch(vi_mag_sq, tij_mag_sq_tensor)
        
#         # Sum of branch admittance and charging admittance
#         Y_total = torch.stack([
#             Yij[idx, 0] + Yijc[idx, 0],
#             Yij[idx, 1] + Yijc[idx, 1]
#         ]).expand(batch_size, -1)
        
#         term1 = complex_mult_batch(complex_conj_batch(Y_total), vi_over_tij_sq)
        
#         # Second term calculations
#         vivj = complex_mult_batch(vi, complex_conj_batch(vj))
#         term2 = complex_mult_batch(
#             complex_conj_batch(Yij[idx].expand(batch_size, -1)),
#             complex_div_batch(vivj, Tij[idx].expand(batch_size, -1))
#         )
        
#         # Total branch flow Sij
#         Sij = term1 - term2
#         branch_flows[(i, j, idx)] = Sij
        
#         # Reverse flow calculations
#         vj_mag_sq = torch.sum(vj**2, dim=-1, keepdim=True)
#         vj_mag_sq = torch.cat([vj_mag_sq, torch.zeros_like(vj_mag_sq)], dim=-1)
        
#         term1_ji = complex_mult_batch(complex_conj_batch(Y_total), vj_mag_sq)
#         vjvi = complex_mult_batch(complex_conj_batch(vi), vj)
#         term2_ji = complex_mult_batch(
#             complex_conj_batch(Yij[idx].expand(batch_size, -1)),
#             complex_div_batch(vjvi, complex_conj_batch(Tij[idx].expand(batch_size, -1)))
#         )
        
#         Sji = term1_ji - term2_ji
#         branch_flows[(j, i, idx)] = Sji
    
    
#     # Aggregate generator power for each node (vectorized)
#     for i in range(num_nodes):
#         for index, (from_bus, to_bus) in enumerate(branches):
#             if from_bus == i:
#                 generator_power[:, i] += branch_flows[(from_bus, to_bus, index)]
#             if to_bus == i:
#                 generator_power[:, i] += branch_flows[(to_bus, from_bus, index)]
    
#     generator_power += demand + shunt_power
    
#     return generator_power, branch_flows

In [ ]:
def convert_to_complex_voltage(voltage_tensor):
    # Extract angle and magnitude
    voltage_angle = voltage_tensor[:,0:1]  # In radians
    voltage_magnitude = voltage_tensor[:,1:]
    
    # Calculate real and imaginary parts
    real_voltage = voltage_magnitude * torch.cos(voltage_angle)
    imaginary_voltage = voltage_magnitude * torch.sin(voltage_angle)
    
    return torch.concat((real_voltage,imaginary_voltage),dim=1)

In [ ]:
def convert_to_complex_rectangle(tensor_2d):
    # Extract angle and magnitude
    tensor_mag = tensor_2d[:,0:1]  # In radians
    tensor_angle = tensor_2d[:,1:]
    
    # Calculate real and imaginary parts
    real_tensor = tensor_mag * torch.cos(tensor_angle)
    imaginary_tensor = tensor_mag * torch.sin(tensor_angle)
    
    return torch.concat((real_tensor,imaginary_tensor),dim=1)

In [ ]:
def calculate_generator_power(
    demand: torch.Tensor, 
    voltage: torch.Tensor,  
    branches: list,  
    Yks: torch.Tensor,  
    Yij: torch.Tensor,  
    Yijc: torch.Tensor,  
    Tij: torch.Tensor,  
) -> torch.Tensor:
    num_nodes = voltage.shape[0]
    generator_power = torch.zeros_like(demand)
    
    # Helper function for complex multiplication
    def complex_mult(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        return torch.stack([
            a[0] * b[0] - a[1] * b[1],
            a[0] * b[1] + a[1] * b[0]
        ])

    # Helper function for complex conjugate
    def complex_conj(x: torch.Tensor) -> torch.Tensor:
        return torch.stack([x[0], -x[1]])

    # Helper function for complex division
    def complex_div(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        denominator = b[0]**2 + b[1]**2
        return torch.stack([
            (a[0] * b[0] + a[1] * b[1]) / denominator,
            (a[1] * b[0] - a[0] * b[1]) / denominator
        ])
    
    # Calculate shunt power terms for each node
    shunt_power = torch.zeros_like(demand)
    for i in range(num_nodes):
        v_mag_sq = voltage[i, 0]**2 + voltage[i, 1]**2
        v_mag_sq_tensor = torch.tensor([v_mag_sq, 0.0])
        shunt_power[i] = complex_mult(complex_conj(Yks[i]), v_mag_sq_tensor)
    # print('the shunt powers are: ', shunt_power)
    
    # Calculate branch flows
    branch_flows = {} 
    
    for idx, (i, j) in enumerate(branches):
        # Get complex voltage at both ends
        vi = voltage[i]
        vj = voltage[j]
        
        # First term of Sij
        vi_mag_sq = vi[0]**2 + vi[1]**2
        vi_mag_sq_tensor = torch.tensor([vi_mag_sq, 0.0], dtype=torch.float64)
        tij_mag_sq = Tij[idx, 0]**2 + Tij[idx, 1]**2
        tij_mag_sq_tensor = torch.tensor([tij_mag_sq, 0.0], dtype=torch.float64)
        vi_over_tij_sq = complex_div(vi_mag_sq_tensor, tij_mag_sq_tensor)
        
        # Sum of branch admittance and charging admittance
        Y_total = torch.stack([
            Yij[idx, 0] + Yijc[idx, 0],
            Yij[idx, 1] + Yijc[idx, 1]
        ])
        term1 = complex_mult(complex_conj(Y_total), vi_over_tij_sq)
        
        # Second term of Sij
        vivj = complex_mult(vi, complex_conj(vj))
        term2 = complex_mult(
            complex_conj(Yij[idx]),
            complex_div(vivj, Tij[idx])
        )
        
        # Total branch flow Sij
        Sij = term1 - term2
        branch_flows[(i, j, idx)] = Sij
        # print(f'the branch flows for {i} , {j} in forward direction are: ', Sij)
        
        # Reverse flow Sji
        term1 = complex_mult(complex_conj(Y_total), vi_over_tij_sq)
        vj_mag_sq = vj[0]**2 + vj[1]**2
        vj_mag_sq_tensor = torch.tensor([vj_mag_sq, 0.0], dtype=torch.float64)
        
        term1_ji = complex_mult(complex_conj(Y_total), vj_mag_sq_tensor)
        vjvi = complex_mult(complex_conj(vi), vj)
        term2_ji = complex_mult(
            complex_conj(Yij[idx]),
            complex_div(vjvi, complex_conj(Tij[idx]))
        )
        Sji = term1_ji - term2_ji
        branch_flows[(j, i, idx)] = Sji
        # print(f'the branch flows for {j} , {i} in reverse direction are: ', Sji)

    # Aggregate generator power for each node
    for i in range(num_nodes):
        for index, (from_bus, to_bus) in enumerate(branches): 
            if from_bus == i:
                generator_power[i] += branch_flows[(from_bus, to_bus, index)]
            if to_bus == i:
                generator_power[i] += branch_flows[(to_bus, from_bus, index)]
        generator_power[i] += demand[i] + shunt_power[i]
    
    return generator_power, branch_flows


In [ ]:
# conductance_susceptance = np.concatenate((edge_g, edge_b), axis=1)
# conductance_susceptance = torch.tensor(conductance_susceptance).to(torch.float32)
# charging_susceptance = torch.zeros_like(conductance_susceptance).to(torch.float32)
# charging_susceptance[:,1:] =  edge_inputs[:,2:3].to('cpu')
# Tij = edge_inputs[:,9:].to('cpu').to(torch.float32)
# Tij[:grid_ac_line_features.shape[1],0:1] = 1.0
# Tij_rec = convert_to_complex_rectangle(Tij)

In [ ]:
load_demand = load_demand.to(torch.float32)

In [ ]:
Yks = torch.zeros(test_grid_shunt[test_indices].shape[0], system_size, test_grid_shunt[test_indices].shape[-1]).to(torch.float32)

Yks[:,shunt_indices,:] = torch.tensor(test_grid_shunt[test_indices])

Yks = Yks[:,:, [1, 0]]

In [ ]:
# injection_balance,branch_flows = calculate_only_branch_flows(load_demand.to('cpu'),complex_v.to('cpu'),branch_list,Yks,conductance_susceptance,charging_susceptance,Tij_rec)

In [ ]:
load_input = load_demand.cpu()

In [ ]:
voltage_predictions = v_predictions.cpu()

In [ ]:
Gen_Powers = []

Branch_Flows = []

In [ ]:
for r in range(len(test_branch_list)):
    Branches = test_branch_list[r]
    ac_active_mask   = test_branch_status[r,line_indices,0] != 0.0  # (17,) bool
    xfmr_active_mask = test_branch_status[r,transformer_indices,0] != 0.0  # (3,)  bool
    present_edge_input = compute_homogenous_edges(Branches,test_grid_ac_line_features[r][ac_active_mask],test_grid_transformer_features[r][xfmr_active_mask])
    edge_g, edge_b = compute_gandb(present_edge_input)
    conductance_susceptance = torch.cat((edge_g, edge_b), dim=1)
    conductance_susceptance = conductance_susceptance.to('cpu')
    charging_susceptance = torch.zeros_like(conductance_susceptance)
    charging_susceptance[:,1:] =  present_edge_input[:,2:3].to('cpu')
    Tij = present_edge_input[:,9:].to('cpu')
    Tij_rec = convert_to_complex_rectangle(Tij)
    complex_v = convert_to_complex_voltage(voltage_predictions[r])
    test_shunt = Yks[r].to('cpu')
    load= load_input[r].to('cpu')
    gen_injection,branch_flows = calculate_generator_power(load,complex_v,Branches,test_shunt,conductance_susceptance,charging_susceptance,Tij_rec)
    Gen_Powers.append(gen_injection)
    Branch_Flows.append(branch_flows)

In [ ]:
generator_power_balance = torch.stack(Gen_Powers,dim=0)

In [ ]:
test_generator_indices = []
for y in range(len(separate_p_targets)):
    gen_active_mask  = test_gen_status[y,:,0] != 0.0
    active_gen_indices   = generator_indices[gen_active_mask]
    test_generator_indices.append(active_gen_indices)

In [ ]:
def build_generator_input_tensor(
    grid_generators: list,
    generator_indices: list,
    system_size: int,
    all_generator_indices: np.array, 
    num_features: int = 11,
) -> torch.Tensor:
    num_samples = len(grid_generators)

    max_slots = max(
        torch.unique(torch.tensor(indices), return_counts=True)[1].max().item()
        for indices in generator_indices
    )

    output = torch.zeros(num_samples, system_size, max_slots, num_features)

    for sample_idx in range(num_samples):
        generators        = grid_generators[sample_idx]    # always 5 rows
        active_node_set   = set(generator_indices[sample_idx].tolist())  # e.g. {0, 2, 5, 7}
        next_slot_per_node = torch.zeros(system_size, dtype=torch.long)

        for row_idx, node_idx in enumerate(all_generator_indices):
            if node_idx not in active_node_set:
                continue  # this generator is outaged, skip it

            slot = next_slot_per_node[node_idx].item()
            output[sample_idx, node_idx, slot] = torch.tensor(generators[row_idx])
            next_slot_per_node[node_idx] += 1

    return output


In [ ]:
# --- Usage ---
test_generator_inputs = build_generator_input_tensor(
    grid_generators=test_grid_generator,
    generator_indices=test_generator_indices,
    system_size=system_size,
    all_generator_indices= generator_indices
)

In [ ]:
# First, determine the maximum number of generators at any single location
max_generators_per_node = 0
for k in range(len(separate_p_targets)):
    unique_indices, counts = torch.unique(torch.tensor(test_generator_indices[k]), return_counts=True)
    max_generators_per_node = max(max_generators_per_node, counts.max().item())

# Create tensor with additional dimension to hold multiple generators per node
test_powers = torch.zeros((len(separate_p_predictions), system_size, max_generators_per_node, separate_p_predictions[0].shape[-1]))

# Fill the tensor with generator values
for k in range(len(separate_p_predictions)):
    instant_power = test_powers[k]
    instant_gen_indices = test_generator_indices[k]
    
    # Track how many generators we've already seen at each node
    node_counts = torch.zeros(system_size, dtype=torch.long)
    
    # Assign each generator output to its proper location
    for i, idx in enumerate(instant_gen_indices):
        # Place this generator's output in the next available slot for this node
        generator_slot = node_counts[idx]
        instant_power[idx, generator_slot] = separate_p_predictions[k][i]
        # Increment the count for this node
        node_counts[idx] += 1

In [ ]:
# First, determine the maximum number of generators at any single location
max_generators_per_node = 0
for k in range(len(separate_p_targets)):
    unique_indices, counts = torch.unique(torch.tensor(test_generator_indices[k]), return_counts=True)
    max_generators_per_node = max(max_generators_per_node, counts.max().item())

# Create tensor with additional dimension to hold multiple generators per node
target_powers = torch.zeros((len(separate_p_targets), system_size, max_generators_per_node, separate_p_targets[0].shape[-1]))

# Fill the tensor with generator values
for k in range(len(separate_p_targets)):
    instant_power = target_powers[k]
    instant_gen_indices = test_generator_indices[k]
    
    # Track how many generators we've already seen at each node
    node_counts = torch.zeros(system_size, dtype=torch.long)
    
    # Assign each generator output to its proper location
    for i, idx in enumerate(instant_gen_indices):
        # Place this generator's output in the next available slot for this node
        generator_slot = node_counts[idx]
        instant_power[idx, generator_slot] = separate_p_targets[k][i]
        # Increment the count for this node
        node_counts[idx] += 1

In [ ]:
def compute_optimality(test_inputs, predictions, test_objective):

    test_inputs = test_inputs.cpu()
    test_objective = test_objective.cpu()

    c2 = test_inputs[:,:,:,0:1]
    c1 = test_inputs[:,:,:,1:2]
    c0 = test_inputs[:,:,:,2:3]

    # Get relevant output dimensions (zero-indexed)
    p_gens = predictions[:,:,:,0:1] # select on Pgs for generators
  

    print('number of nonzero  c2 is ',  torch.count_nonzero(c2[0]))
    print('the shape of p_gen is ', p_gens.shape)
    
    # Compute node-wise metrics
    system_metrics = c2 * (p_gens ** 2) + c1 * p_gens + c0
    print('the shape of system_metrics is ', system_metrics.shape)


    model_obj = torch.sum(system_metrics, dim=(1,2,3))

    print('the shape of test objective is ', test_objective.shape)
    print('the shape of model objective is ', model_obj.shape)

    print(f'average model objective is {model_obj.mean()}')
    print(f'average IPOPT objective is {test_objective.mean()}')

    optimality_gap = (model_obj / test_objective) * 100

    
    return optimality_gap.mean()

In [ ]:
test_obj = torch.tensor(test_solution_objective[test_indices])
test_generator_cost = test_generator_inputs[:,:,:,8:11]

In [ ]:
opt_gap = compute_optimality(test_generator_cost, test_powers, test_obj)

In [ ]:
print('optimality gap now is ', opt_gap)

In [ ]:
def calculate_angle_differences(angles, edges):
  
    # Ensure angles are a NumPy array
    angles = np.asarray(angles)    
    # Initialize array to store angle differences
    angle_differences = np.zeros(len(edges), dtype=np.float32)
    
    # Calculate angle differences for each edge
    for i, (node1, node2) in enumerate(edges):
        angle_differences[i] = angles[node2] - angles[node1]
    
    return angle_differences

In [ ]:
##### time to evaluate constrain satisfactions

In [ ]:
predicted_angle_differences = []
predicted_angles = v_predictions[:,:,0]

for j in range(v_predictions.shape[0]):
    
    branch_for_this = test_branch_list[j]
    angle_differences = calculate_angle_differences(predicted_angles[j],branch_for_this)
    angle_differences = torch.tensor(angle_differences)
    predicted_angle_differences.append(angle_differences)
    

In [ ]:
true_angle_differences = []
true_angles = v_targets[:,:,0]

for j in range(v_targets.shape[0]):
    
    branch_for_this = test_branch_list[j]
    angle_differences = calculate_angle_differences(true_angles[j],branch_for_this)
    angle_differences = torch.tensor(angle_differences)
    true_angle_differences.append(angle_differences)

In [ ]:
predicted_angle_differences = torch.cat(predicted_angle_differences)
true_angle_differences = torch.cat(true_angle_differences)

In [ ]:
# voltage angle difference bound 


angle_diff_upper = torch.full(predicted_angle_differences.shape, 0.5236)
angle_diff_lower = torch.full(predicted_angle_differences.shape, -0.5236)

# Calculate violations
lower_angle_violations = torch.clamp(angle_diff_lower - predicted_angle_differences, min=0)  # Positive if below lower bound
upper_angle_violations = torch.clamp(predicted_angle_differences - angle_diff_upper, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
angle_diff_violations = lower_angle_violations + upper_angle_violations

print('max voltage angle difference violation is : ',angle_diff_violations.max() )
print('average voltage angle difference violation is : ',angle_diff_violations.mean())

In [ ]:
test_buses = torch.tensor(np.array(test_grid_bus))

In [ ]:
## voltage magnitude bound

vmin = test_buses[:,:,2:3].to('cpu')

vmax = test_buses[:,:,3:4].to('cpu')

lower_vmag_violation = torch.clamp(vmin - v_predictions[:,:,1:2], min=0)  # Positive if below lower bound

upper_vmag_violation = torch.clamp(v_predictions[:,:,1:2] - vmax, min=0)  # Positive if above upper bound

vmag_violations = lower_vmag_violation + upper_vmag_violation

print('max voltage magnitude violation is : ',vmag_violations.max() )
print('average voltage magnitude violation is : ',vmag_violations.mean() )

In [ ]:
# def build_generator_input_tensor(
#     grid_generators: list,
#     generator_indices: list,
#     system_size: int,
#     all_generator_indices: np.array, 
#     num_features: int = 11,
# ) -> torch.Tensor:
#     num_samples = len(grid_generators)

#     max_slots = max(
#         torch.unique(torch.tensor(indices), return_counts=True)[1].max().item()
#         for indices in generator_indices
#     )

#     output = torch.zeros(num_samples, system_size, max_slots, num_features)

#     for sample_idx in range(num_samples):
#         generators        = grid_generators[sample_idx]    # always 5 rows
#         active_node_set   = set(generator_indices[sample_idx].tolist())  # e.g. {0, 2, 5, 7}
#         next_slot_per_node = torch.zeros(system_size, dtype=torch.long)

#         for row_idx, node_idx in enumerate(all_generator_indices):
#             if node_idx not in active_node_set:
#                 continue  # this generator is outaged, skip it

#             slot = next_slot_per_node[node_idx].item()
#             output[sample_idx, node_idx, slot] = torch.tensor(generators[row_idx])
#             next_slot_per_node[node_idx] += 1

#     return output


# # --- Usage ---
# test_generator_inputs = build_generator_input_tensor(
#     grid_generators=test_grid_generator,
#     generator_indices=test_generator_indices,
#     system_size=system_size,
#     all_generator_indices= generator_indices
# )

In [ ]:
# Gen active power bounds 
pmin = test_generator_inputs[:,:,:,2:3].to('cpu')

pmax = test_generator_inputs[:,:,:,3:4].to('cpu')


p_gens = test_powers[:,:,:,0:1]


lower_pgen_violations = torch.clamp(pmin - p_gens, min=0)  # Positive if below lower bound
upper_pgen_violations = torch.clamp(p_gens - pmax, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
pgen_violations = lower_pgen_violations + upper_pgen_violations

print('max gen active power violation is : ',pgen_violations.max())
print('average active power violation is : ',pgen_violations.mean())

In [ ]:
# Gen reactive power bounds 
qmin = test_generator_inputs[:,:,:,5:6].to('cpu')

qmax = test_generator_inputs[:,:,:,6:7].to('cpu')


q_gens = test_powers[:,:,:,1:2]


lower_qgen_violations = torch.clamp(qmin - q_gens, min=0)  # Positive if below lower bound
upper_qgen_violations = torch.clamp(q_gens - qmax, min=0)  # Positive if above upper bound

# Combine violations into a single tensor
qgen_violations = lower_qgen_violations + upper_qgen_violations

print('max gen reactive power violation is : ',qgen_violations.max())
print('average reactive power violation is : ',qgen_violations.mean())

In [ ]:
# compute the power magnitudes

# Separate real and imaginary parts
def convert_to_power_magnitude(power_flow):
    
    real = power_flow[..., 0]  
    imag = power_flow[..., 1]  

    # Compute the magnitudes
    magnitudes = torch.sqrt(real**2 + imag**2) 


    result_tensor = magnitudes.unsqueeze(-1)

    return result_tensor

In [ ]:
for_flow_vio = []
rev_flow_vio = []

In [ ]:
long_term_line_rating = []

for k in range(len(test_grid_ac_line_features)):
    
    ac_active_mask   = test_branch_status[k,line_indices,0] != 0.0  # (17,) bool
    xfmr_active_mask = test_branch_status[k,transformer_indices,0] != 0.0  # (3,)  bool
    all_lines = test_grid_ac_line_features[k]
    line_thermal_ratings = torch.tensor(all_lines[:,6:7])
    line_thermal_ratings = line_thermal_ratings[ac_active_mask]
    all_transformers = test_grid_transformer_features[k]
    trans_thermal_ratings = torch.tensor(all_transformers[:,4:5])
    trans_thermal_ratings = trans_thermal_ratings[xfmr_active_mask]
    all_edge_ratings = torch.cat((line_thermal_ratings, trans_thermal_ratings), dim=0)
    long_term_line_rating.append(all_edge_ratings)



In [ ]:
for k in range(len(Branch_Flows)):
    flow_branch = test_branch_list[k]
    forward_keys = [(i, j, index) for index, (i,j) in enumerate(flow_branch)]
    reverse_keys = [(j, i, index) for index, (i,j) in enumerate(flow_branch)]
    forward_branch_flows = {key: Branch_Flows[k][key] for key in forward_keys if key in Branch_Flows[k]}
    reverse_branch_flows = {key: Branch_Flows[k][key] for key in reverse_keys if key in Branch_Flows[k]}


    forward_power_flows_list = [tensor for tensor in forward_branch_flows.values()]
    
    
    # Step 2: Concatenate tensors along the second axis (dim=1)
    forward_power_flows = torch.cat(forward_power_flows_list, dim=0).reshape(-1,2)



    reverse_power_flows = [tensor for tensor in reverse_branch_flows.values()]
    # Step 2: Concatenate tensors along the second axis (dim=1)
    reverse_power_flows = torch.cat(reverse_power_flows, dim=0).reshape(-1,2)
    forward_flow_magnitude = convert_to_power_magnitude(forward_power_flows)
    reverse_flow_magnitude = convert_to_power_magnitude(reverse_power_flows)
    # Branch flow bounds in forward direction


    branch_flow_limit = long_term_line_rating[k]
    forward_branch_flow = forward_flow_magnitude
    forward_flow_violations = torch.clamp(forward_branch_flow - branch_flow_limit, min=0)  # Positive if above upper bound
    for_flow_vio.append(forward_flow_violations)
    reverse_branch_flow = reverse_flow_magnitude
    reverse_flow_violations = torch.clamp(reverse_flow_magnitude - branch_flow_limit, min=0)  # Positive if above upper bound
    rev_flow_vio.append(reverse_flow_violations)



In [ ]:
for_flow_vio = torch.cat(for_flow_vio)
rev_flow_vio = torch.cat(rev_flow_vio)

In [ ]:
print('max forward power flow violation is : ',for_flow_vio.max() )
print('average  forward power flow violation is : ',for_flow_vio.mean() )

In [ ]:
print('max reverse power flow violation is : ', rev_flow_vio.max() )
print('average reverse power flow violation is : ', rev_flow_vio.mean() )

In [ ]:
test_power_per_node = torch.sum(test_powers, dim=2)

In [ ]:
true_power_per_node = torch.sum(target_powers, dim=2)

In [ ]:
## Evaluate power balance contraint violations
real_power_balance_mismatches = generator_power_balance[:,:,0] - test_power_per_node[:,:,0]


print('max active power balance mismatch is : ', real_power_balance_mismatches.max() )
print('average active power balance mismatch is : ', real_power_balance_mismatches.mean())

In [ ]:
reactive_power_balance_mismatches = generator_power_balance[:,:,1] - test_power_per_node[:,:,1]


print('max reactive power balance mismatch is : ', reactive_power_balance_mismatches.max() )
print('average reactive power balance mismatch is : ', reactive_power_balance_mismatches.mean())

In [ ]:
#create table to save important metrics
columns=["metric", "value"]
model_metrics_table = wandb.Table(columns=columns)

In [ ]:
model_metrics_table.add_data("optimality gap", opt_gap)
model_metrics_table.add_data("max voltage angle difference violation", angle_diff_violations.max())
model_metrics_table.add_data("average voltage angle difference violation", angle_diff_violations.mean())
model_metrics_table.add_data("max voltage magnitude violation", vmag_violations.max())
model_metrics_table.add_data("average voltage magnitude violation", vmag_violations.mean())
model_metrics_table.add_data("max gen active power violation", pgen_violations.max())
model_metrics_table.add_data("average gen active power violation", pgen_violations.mean())
model_metrics_table.add_data("max gen reactive power violation", qgen_violations.max())
model_metrics_table.add_data("average gen reactive power violation", qgen_violations.mean())
model_metrics_table.add_data("max forward power flows violation", forward_flow_violations.max())
model_metrics_table.add_data("average forward power flows violation", forward_flow_violations.mean())
model_metrics_table.add_data("max reverse power flows violation", reverse_flow_violations.max())
model_metrics_table.add_data("average reverse power flows violation", reverse_flow_violations.mean())
model_metrics_table.add_data("max active power balance mismatch", real_power_balance_mismatches.max())
model_metrics_table.add_data("average active power balance mismatch", real_power_balance_mismatches.mean())
model_metrics_table.add_data("max reactive power balance mismatch", reactive_power_balance_mismatches.max())
model_metrics_table.add_data("average reactive power balance mismatch", reactive_power_balance_mismatches.mean())
wandb.log({"model_metrics_table" : model_metrics_table})

In [ ]:
wandb.finish()

In [ ]:
torch.cuda.empty_cache()
gc.collect()